So to handle something called as long sequences we had something called as RNN. 

they way RNN works it would pass 

input text 1 + hidden state 0 

generate hidden state 1

input text 2 + hidden state 1

and so on. RNN can't directly access earlier hidden states from encoder during the decoding states, because of which there is a pressure built up on the final hidden state which it would eventually fup so long sequences are hard to handle.

RNN Works for small text but long text it would struggle.


This is where attention mechanism come into picture.

But before the attention is all you need got famous in 2014, there was this discovery to modifie encoder and decoder of RNN such that decoder can selectively access different parts of input sequence at each decoding step.



Soon people realized that RNNS are not required for DNN for natural language processing and researcher proposed transformer architecture.


Self attention 

this doesn't map like input to output like in earlier where we were doing translation. 

the main goal for all this is to generate next word, so what self attention does is. 

it will attend each word individually along with correlation to another words.

like how each words correlate to every other words.


Here "self" refers to the mechanism ability to compute attention weight by relation different position in a single input sequence. here we learn relationship on various parts of input

Goal of any attention mechanism is to create context vector

So all this token embeddings are just vectors right. 

what we can do is just plot them in the 3d space there are some vector who would be closer and some would be further. 


when it comes to finding similarity between 2 words we make use of dot product why is that because, formula is |a||b| cos tetha. 

cos 0 => 1 
cos 90 => 0

so if dot product of 2 vector is greater value which means they are closer to each other in 3 dimensional space based of that we assign the weights.

In [13]:
import torch

In [15]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [16]:
query = inputs[1]

attention_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attention_scores_2[i] = torch.dot(x_i, query)

attention_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

we would normalize the attention score because it 
becomes interpretable and also maintaing training stability
attention score doesn't sum up to 2 
attention weight sum up to 1
Essentially both are same just different representation

usually we do softmax 

because problem with summation and normalizing is just outllier creates such a big issue, for eg. 1,2,3,400

softmax

e^x1/sum ... e^xn/sum

pytorch 

e^x1-max/sum...e^xn-max/sum

In [ ]:

attention_weight_temp = attention_scores_2 / attention_scores_2.sum()
attention_weight_temp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

def softmax_torch(x):
    return torch.exp(x - torch.max(x)) / torch.sum(torch.exp(x - torch.max(x)))


tensor([0., 0., 0., 1.])

In [22]:
attention_weights_2_naive = softmax_naive(attention_scores_2)

In [25]:
attention_weights_2_torch = softmax_torch(attention_scores_2)
attention_weights_2_torch_2 = torch.softmax(attention_scores_2, dim=0)

print(attention_weights_2_naive)
print(attention_weights_2_torch)
print(attention_weights_2_torch_2)



tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


After coming the atttention weight, we can get the context vector

we calculate context vector by multiplying the embedded input tokens with the corresponding attention weights and then summing the reultant vectors.

Context vector is improved state of embedding vector